<a href="https://colab.research.google.com/github/abdo0412/decentralized-insurance-fraud-detection/blob/main/notebooks/01_vision_cnn_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 7 — Part A: Vision Agent CNN Training (Google Colab / Kaggle GPU)

**Owner:** Person A (Vision Specialist)  
**Model:** ResNet50 (`IMAGENET1K_V2`) Two-Stage Transfer Learning + Grad-CAM + Held-Out Pool Scoring  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdo0412/decentralized-insurance-fraud-detection/blob/main/notebooks/01_vision_cnn_training.ipynb)

### Instructions:
1. In Google Colab, go to **Runtime → Change runtime type → T4 GPU**.
2. Run all cells from top to bottom (~5–8 minutes total).
3. The last cell zips and downloads your trained checkpoint (`models/vision_resnet50_v1.pt`), evaluation plots (`report/figures/`), and Person C's held-out predictions (`data/processed/vision_predictions.csv`).

In [ ]:
# 1. Clone the repository & verify GPU
import os
import torch

REPO_URL = "https://github.com/abdo0412/decentralized-insurance-fraud-detection.git"
REPO_DIR = "decentralized-insurance-fraud-detection"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}
!git pull origin main

print("PyTorch version:", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device name    :", torch.cuda.get_device_name(0))

In [ ]:
# 2. Download the Kaggle Car Damage Severity Dataset into data/raw/vehicle_damage
import io
import os
import urllib.request
import zipfile
from getpass import getpass

os.makedirs("data/raw/vehicle_damage", exist_ok=True)

# Uses KAGGLE_API_TOKEN env var if set, or prompts securely
kaggle_token = os.environ.get("KAGGLE_API_TOKEN", "").strip()
if not kaggle_token:
    kaggle_token = getpass("Paste your Kaggle API Token (starts with KGAT_...): ").strip()

url = "https://www.kaggle.com/api/v1/datasets/download/prajwalbhamere/car-damage-severity-dataset"
req = urllib.request.Request(url, headers={"Authorization": f"Bearer {kaggle_token}"})
print("Downloading dataset from Kaggle...")
with urllib.request.urlopen(req) as resp:
    data = resp.read()
print(f"Downloaded {len(data) / (1024*1024):.2f} MB. Extracting...")
with zipfile.ZipFile(io.BytesIO(data)) as zf:
    zf.extractall("data/raw/vehicle_damage")
print("Extraction complete!")

In [ ]:
# 3. Verify the Leak-Free Split Manifests (already committed in Git, or re-verify)
import pandas as pd

held_out_df = pd.read_csv("data/processed/held_out_images.csv")
train_df = pd.read_csv("data/processed/cnn_splits/train.csv")
val_df = pd.read_csv("data/processed/cnn_splits/val.csv")
test_df = pd.read_csv("data/processed/cnn_splits/test.csv")

print(f"Held-Out Pool (for Person C, untouched by CNN): {len(held_out_df)} images")
print(f"CNN Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Assert zero overlap between CNN pool and Held-Out pool
cnn_paths = set(train_df["image_path"]).union(val_df["image_path"]).union(test_df["image_path"])
held_out_paths = set(held_out_df["image_path"])
assert len(cnn_paths.intersection(held_out_paths)) == 0, "DATA LEAKAGE DETECTED!"
print("Zero data leakage verified!")

In [ ]:
# 4. Train & Evaluate the Vision CNN (Stage 1 Warmup + Stage 2 Fine-Tuning)
!python scripts/train_vision_cnn.py \
    --splits-dir data/processed/cnn_splits \
    --backbone resnet50 \
    --epochs-warmup 5 \
    --epochs-finetune 12 \
    --batch-size 32 \
    --lr-warmup 1e-3 \
    --lr-finetune 1e-4

In [ ]:
# 5. Display Test Metrics, Confusion Matrix, and Grad-CAM Examples
import json
from IPython.display import Image as IPyImage, display

with open("report/figures/vision_metrics.json", "r") as f:
    metrics = json.load(f)

print(f"Test Accuracy : {metrics['test_accuracy']:.4f}")
print(f"Test Macro F1 : {metrics['test_macro_f1']:.4f}")

display(IPyImage(filename="report/figures/vision_confusion_matrix.png"))
display(IPyImage(filename="report/figures/vision_gradcam_examples.png"))

In [ ]:
# 6. Score the 490 Held-Out Photos for Person C's Decision Agent
!python scripts/predict_held_out.py \
    --claims-csv data/processed/held_out_images.csv \
    --weights models/vision_resnet50_v1.pt \
    --backbone resnet50 \
    --output-csv data/processed/vision_predictions.csv

preds_df = pd.read_csv("data/processed/vision_predictions.csv")
preds_df.head(10)

In [ ]:
# 7. Package & Download All Part A Deliverables to Your Computer
!zip -r part_a_deliverables.zip \
    models/vision_resnet50_v1.pt \
    report/figures/vision_confusion_matrix.png \
    report/figures/vision_gradcam_examples.png \
    report/figures/vision_metrics.json \
    data/processed/vision_predictions.csv

try:
    from google.colab import files
    files.download("part_a_deliverables.zip")
except ImportError:
    print("Saved part_a_deliverables.zip in the working directory!")